# NB01 — Airbnb Calendar ETL

**Technologies:** Microsoft Fabric | PySpark | Delta Lake  
**Data Layer:** Raw CSV → Silver

## 1. Objective

Clean and validate Airbnb calendar data to analyse **advertised availability** over a 90-day period.

The processed data supports downstream availability analysis by listing, date and LGA.

## 2. Data Sources & Outputs

| Type | Source |
|---|---|
| Raw Input | `Files/calendar.csv` |
| Reference | `dbo.silver_listings` |
| Silver Output | `dbo.silver_calendar_90d` |
| Downstream Gold | `dbo.fact_availability`, `dbo.dim_date` |


**Silver Data Grain:** One record per listing per date (listing_id + date).

**Downstream Gold Data Grain:** fact_availability preserves the Listing-Date grain, while dim_date contains one unique record per calendar date.

## 3. ETL Process

1. **Ingestion & Cleaning:** Read CSV, convert data types and remove invalid records.
2. **Dynamic Date Filtering:** Use the latest Listings snapshot date and the following 89 days (90 days inclusive).
3. **Availability Transformation:** Convert `available` (`t/f`) into `is_available` (`1/0`).
4. **Data Quality Validation:** Check record counts, duplicate Listing-Date keys and complete 90-day coverage.
5. **Silver Storage:** Write validated data to `dbo.silver_calendar_90d` in Delta format.

## 4. Date Selection & Data Quality

**Why 18 September 2026?**

The earlier 90-day window (17 Sep – 15 Dec) contained incomplete Listing-Date records.

| Metric | 17 Sep – 15 Dec | 18 Sep – 16 Dec |
|---|---:|---:|
| Expected Records | 2,314,350 | 2,314,350 |
| Actual Records | 2,292,990 | 2,314,350 |
| Missing Records | 21,360 | 0 |

**Design Decision:** Use 18 September, the latest Listings snapshot date, to achieve complete 90-day coverage. The start date is derived dynamically rather than hard-coded.

## 5. Business Definitions

**Advertised Availability (`is_available`)**

| Value | Meaning |
|---|---|
| `1` | Listing advertised as available on the specified date |
| `0` | Listing not advertised as available on the specified date |

Availability does not guarantee a successful booking, and unavailable dates do not necessarily indicate confirmed bookings.

**Advertised availability ≠ actual occupancy or demand.**

## 6. Validated Output

| Metric | Result |
|---|---:|
| Unique Listings | 25,715 |
| Analysis Period | 18 Sep – 16 Dec 2026 (90 days) |
| Listing-Date Records | 2,314,350 |
| Duplicate Keys | 0 |
| Incomplete Listing Coverage | 0 |

## 7. Downstream Dependency

**NB02 Listings → NB01 Calendar → NB04 Gold**

NB01 requires the Listings snapshot date. NB04 then uses the validated Silver data to build the Availability Fact Table and Date Dimension, including weekend and public holiday indicators.

In [ ]:
# Import Required Libraries
# Python Standard Library - Date calculations
from datetime import timedelta

# PySpark - Data transformation and validation
from pyspark.sql import functions as F

StatementMeta(, 9ef3ae31-6f5f-4c9d-a264-b9953891438e, 3, Finished, Available, Finished, False)

In [2]:
# Step 1: Read raw Airbnb calendar data

df = (
    spark.read
    .option("header", "true")
    .csv("Files/calendar.csv")
)

# Check columns and data types
df.printSchema()

# Preview the first 5 rows
display(df.limit(5))


StatementMeta(, 9ef3ae31-6f5f-4c9d-a264-b9953891438e, 4, Finished, Available, Finished, False)

root
 |-- listing_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- available: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- maximum_nights: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 3577b208-0905-4b39-9d6c-72ef3207dc12)

In [3]:
# Identify the analysis window using Listings snapshot

#  Step 2A: Clean raw Calendar data

calendar = df.select(
    F.col("listing_id").cast("long").alias("listing_id"),
    F.to_date(F.col("date"), "yyyy-MM-dd").alias("date"),
    F.lower(F.trim(F.col("available"))).alias("available"),
    F.col("minimum_nights").cast("int").alias("minimum_nights"),
    F.col("maximum_nights").cast("int").alias("maximum_nights")
)

# Keep valid Calendar records
calendar = calendar.filter(
    F.col("listing_id").isNotNull() &
    F.col("date").isNotNull() &
    F.col("available").isin("t", "f")
)

print("Cleaned Calendar Rows:", calendar.count())



# Read the cleaned Listings snapshot
listings = spark.table("dbo.silver_listings")

# Find the snapshot date range
snapshot_dates = listings.agg(
    F.min(F.to_date("snapshot_date")).alias("earliest"),
    F.max(F.to_date("snapshot_date")).alias("latest")
).first()

print("Earliest Snapshot:", snapshot_dates["earliest"])
print("Latest Snapshot:", snapshot_dates["latest"])

# Use the latest snapshot date as the analysis anchor
start_date = snapshot_dates["latest"]

assert start_date is not None, "Missing snapshot date!"

# 90 consecutive days, including the start date
end_date = start_date + timedelta(days=89)

# Filter Calendar to the same analysis window
calendar_90d = calendar.filter(
    F.col("date").between(
        F.lit(start_date),
        F.lit(end_date)
    )
)

# Convert t/f to 1/0
calendar_90d = calendar_90d.withColumn(
    "is_available",
    F.when(F.col("available") == "t", 1).otherwise(0)
)

print("Analysis Start:", start_date)
print("Analysis End:", end_date)
print("Calendar Rows:", calendar_90d.count())


StatementMeta(, 9ef3ae31-6f5f-4c9d-a264-b9953891438e, 5, Finished, Available, Finished, False)

Cleaned Calendar Rows: 9385976
Earliest Snapshot: 2026-09-17
Latest Snapshot: 2026-09-18
Analysis Start: 2026-09-18
Analysis End: 2026-12-16
Calendar Rows: 2314350


In [4]:
# Step 3: Data Quality Validation

# Check duplicate Listing-Date keys
duplicates = (
    calendar_90d
    .groupBy("listing_id", "date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check 90-day coverage
unique_dates = calendar_90d.select("date").distinct().count()
unique_listings = calendar_90d.select("listing_id").distinct().count()

actual_rows = calendar_90d.count()
expected_rows = unique_listings * 90

print("Unique Dates:", unique_dates)
print("Unique Listings:", unique_listings)
print("Expected Rows:", expected_rows)
print("Actual Rows:", actual_rows)
print("Duplicate Keys:", duplicates)

assert unique_listings > 0, "No listings found!"
assert unique_dates == 90, "Expected 90 dates!"
assert duplicates == 0, "Duplicate listing-date keys!"
assert actual_rows == expected_rows, "Incomplete calendar data!"

print("Calendar validation passed!")

# Save Silver Calendar
(
    calendar_90d.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.silver_calendar_90d")
)

print("Silver Calendar updated successfully!")



StatementMeta(, 9ef3ae31-6f5f-4c9d-a264-b9953891438e, 6, Finished, Available, Finished, False)

Unique Dates: 90
Unique Listings: 25715
Expected Rows: 2314350
Actual Rows: 2314350
Duplicate Keys: 0
Calendar validation passed!
Silver Calendar updated successfully!
